In [1]:
from semantic_agent import SemanticAgent

agent = SemanticAgent()

## Q1 "How many loads were delivered in the last full month available in the data?"

In [3]:
q1 = "How many loads were delivered in the last full month available in the data?"
result_q1 = agent.answer_question(q1)

print(f"Question: {result_q1['question']}")
print(f"\nSQL:\n{result_q1['generated_sql']}")
print(f"\nAnswer: {result_q1['answer']}")


Question: How many loads were delivered in the last full month available in the data?

SQL:
WITH max_date AS (
  SELECT MAX(delivery_date) AS max_delivery
  FROM analytics.fact_loads
),
last_full_month AS (
  SELECT DATE_TRUNC('month', max_delivery) - INTERVAL '1 month' AS month_start
  FROM max_date
)
SELECT COUNT(*) AS load_count
FROM analytics.fact_loads, last_full_month
WHERE delivery_date >= month_start
  AND delivery_date < month_start + INTERVAL '1 month';

Answer: [{'load_count': 0}]


## Q2 "Which shipper had the highest total book price?"

In [ ]:
q2 = "Which shipper had the highest total book price?"
result_q2 = agent.answer_question(q2)

print(f"Question: {result_q2['question']}")
print(f"\nSQL:\n{result_q2['generated_sql']}")
print(f"\nAnswer: {result_q2['answer']}")

Question: Which shipper had the highest total book price?

SQL:
SELECT ds.shipper_name, SUM(fl.book_price) AS total_book_price
FROM analytics.fact_loads fl
JOIN analytics.dim_shipper ds ON fl.shipper_id = ds.shipper_id
GROUP BY ds.shipper_name
ORDER BY total_book_price DESC
LIMIT 1;

Answer: [{'shipper_name': 'Shipper 1249', 'total_book_price': 1915694.1600000034}]
Correct? Yes


## Q3 "What is the average book price per load by pickup state?"

In [ ]:
q3 = "What is the average book price per load by pickup state?"
result_q3 = agent.answer_question(q3)

print(f"Question: {result_q3['question']}")
print(f"\nSQL:\n{result_q3['generated_sql']}")
print(f"\nAnswer: {result_q3['answer']}")

Question: What is the average book price per load by pickup state?

SQL:
SELECT
    dl.state AS pickup_state,
    AVG(fl.book_price) AS avg_book_price
FROM analytics.fact_loads fl
JOIN analytics.dim_lane dla ON fl.lane_id = dla.lane_id
JOIN analytics.dim_location dl ON dla.source_location_id = dl.location_id
GROUP BY dl.state
ORDER BY avg_book_price DESC;

Answer: [{'pickup_state': 'SC', 'avg_book_price': 2708.9945454545464}, {'pickup_state': 'AR', 'avg_book_price': 2632.866666666667}, {'pickup_state': 'IA', 'avg_book_price': 2469.435348837209}, {'pickup_state': 'DE', 'avg_book_price': 2242.4857142857145}, {'pickup_state': 'MS', 'avg_book_price': 2100.505}, {'pickup_state': 'OR', 'avg_book_price': 2069.6565}, {'pickup_state': 'NC', 'avg_book_price': 2009.9484716157206}, {'pickup_state': 'MO', 'avg_book_price': 1952.0044366197187}, {'pickup_state': 'WI', 'avg_book_price': 1932.873333333333}, {'pickup_state': 'ID', 'avg_book_price': 1923.7248529411766}, {'pickup_state': 'AZ', 'avg_book_p

## Q4 "What are the top 5 lanes by number of delivered loads?"

In [ ]:
q4 = "What are the top 5 lanes by number of delivered loads?"
result_q4 = agent.answer_question(q4)

print(f"Question: {result_q4['question']}")
print(f"\nSQL:\n{result_q4['generated_sql']}")
print(f"\nAnswer: {result_q4['answer']}")

Question: What are the top 5 lanes by number of delivered loads?

SQL:
SELECT
    dl.lane_name,
    COUNT(*) AS delivered_loads
FROM analytics.fact_loads fl
JOIN analytics.dim_lane dl ON fl.lane_id = dl.lane_id
WHERE fl.delivery_date IS NOT NULL
    AND (fl.load_was_cancelled IS NULL OR fl.load_was_cancelled = FALSE)
GROUP BY dl.lane_name
ORDER BY delivered_loads DESC
LIMIT 5;

Answer: [{'lane_name': 'Hawkins,TX -> Roanoke,TX', 'delivered_loads': 882}, {'lane_name': 'Lodi,CA -> Pacific,WA', 'delivered_loads': 150}, {'lane_name': 'Kent,WA -> Spokane,WA', 'delivered_loads': 94}, {'lane_name': 'Henderson,NV -> Tracy,CA', 'delivered_loads': 87}, {'lane_name': 'Taft,CA -> Tracy,CA', 'delivered_loads': 72}]
Correct? Yes


## Q5 "Which carrier moved the most loads into Texas?"

In [ ]:
q5 = "Which carrier moved the most loads into Texas?"
result_q5 = agent.answer_question(q5)

print(f"Question: {result_q5['question']}")
print(f"\nSQL:\n{result_q5['generated_sql']}")
print(f"\nAnswer: {result_q5['answer']}")

Question: Which carrier moved the most loads into Texas?

SQL:
SELECT dc.carrier_name, COUNT(*) AS load_count
FROM analytics.fact_loads fl
JOIN analytics.dim_carrier dc ON fl.carrier_id = dc.carrier_id
JOIN analytics.dim_lane dl ON fl.lane_id = dl.lane_id
JOIN analytics.dim_location loc ON dl.target_location_id = loc.location_id
WHERE loc.state = 'TX'
GROUP BY dc.carrier_name
ORDER BY load_count DESC
LIMIT 1;

Answer: [{'carrier_name': 'Carrier 567581', 'load_count': 188}]
Correct? Yes


## Q6 "How does the average book price compare between intrastate and interstate loads?"

In [ ]:
q6 = "How does the average book price compare between intrastate and interstate loads?"
result_q6 = agent.answer_question(q6)

print(f"Question: {result_q6['question']}")
print(f"\nSQL:\n{result_q6['generated_sql']}")
print(f"\nAnswer: {result_q6['answer']}")

Question: How does the average book price compare between intrastate and interstate loads?

SQL:
SELECT
    CASE 
        WHEN sl.state = tl.state THEN 'Intrastate'
        ELSE 'Interstate'
    END AS load_type,
    COUNT(*) AS load_count,
    AVG(f.book_price) AS avg_book_price
FROM analytics.fact_loads f
JOIN analytics.dim_lane l ON f.lane_id = l.lane_id
JOIN analytics.dim_location sl ON l.source_location_id = sl.location_id
JOIN analytics.dim_location tl ON l.target_location_id = tl.location_id
GROUP BY 
    CASE 
        WHEN sl.state = tl.state THEN 'Intrastate'
        ELSE 'Interstate'
    END;

Answer: [{'load_type': 'Interstate', 'load_count': 3534, 'avg_book_price': 1709.8168902093944}, {'load_type': 'Intrastate', 'load_count': 1821, 'avg_book_price': 584.4859253157565}]
Correct? Yes


## Q7 "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?"

In [ ]:
q7 = "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?"
result_q7 = agent.answer_question(q7)

print(f"Question: {result_q7['question']}")
print(f"\nSQL:\n{result_q7['generated_sql']}")
print(f"\nAnswer: {result_q7['answer']}")

Question: For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?

SQL:
WITH top_shipper AS (
    SELECT fl.shipper_id
    FROM analytics.fact_loads fl
    WHERE fl.delivery_date IS NOT NULL
      AND fl.load_was_cancelled IS NOT TRUE
    GROUP BY fl.shipper_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
SELECT
    DATE_TRUNC('month', fl.delivery_date) AS delivery_month,
    COUNT(*) AS delivered_loads
FROM analytics.fact_loads fl
JOIN top_shipper ts ON fl.shipper_id = ts.shipper_id
WHERE fl.delivery_date IS NOT NULL
  AND fl.load_was_cancelled IS NOT TRUE
GROUP BY DATE_TRUNC('month', fl.delivery_date)
ORDER BY delivery_month;

Answer: [{'delivery_month': datetime.datetime(2024, 1, 1, 0, 0), 'delivered_loads': 127}, {'delivery_month': datetime.datetime(2024, 2, 1, 0, 0), 'delivered_loads': 108}, {'delivery_month': datetime.datetime(2024, 3, 1, 0, 0), 'delivered_loads': 178}, {'delivery_month': datetime.datetime(2024, 4, 1, 0, 0),

## Q8 "Among lanes with at least 10 delivered loads, which had the highest average book price?"

In [10]:
q8 = "Among lanes with at least 10 delivered loads, which had the highest average book price?"
result_q8 = agent.answer_question(q8)

print(f"Question: {result_q8['question']}")
print(f"\nSQL:\n{result_q8['generated_sql']}")
print(f"\nAnswer: {result_q8['answer']}")

Question: Among lanes with at least 10 delivered loads, which had the highest average book price?

SQL:
SELECT
    dl.lane_name,
    AVG(fl.book_price) AS avg_book_price,
    COUNT(*) AS delivered_load_count
FROM analytics.fact_loads fl
JOIN analytics.dim_lane dl ON fl.lane_id = dl.lane_id
WHERE fl.delivery_date IS NOT NULL
    AND (fl.load_was_cancelled IS NULL OR fl.load_was_cancelled = FALSE)
GROUP BY dl.lane_name
HAVING COUNT(*) >= 10
ORDER BY avg_book_price DESC
LIMIT 1;

Answer: [{'lane_name': 'Stockton,CA -> Parrish,FL', 'avg_book_price': 6800.0, 'delivered_load_count': 11}]


## Q9 (custom) "Which carrier-lane combinations charge at least double the lane's average source price?"

Business reason: `source_price` is what Loadsmart pays the carrier (cost), so a carrier charging 2x+ the lane average on a given lane is decreasing margin (`pnl = book_price - source_price`) and is a candidate for renegotiation or replacement on that lane.

In [11]:
q9 = "Which carrier-lane combinations have a source price at least double the lane's average source price, considering only non-cancelled loads?"
result_q9 = agent.answer_question(q9)

print(f"Question: {result_q9['question']}")
print(f"\nSQL:\n{result_q9['generated_sql']}")
print(f"\nAnswer: {result_q9['answer']}")

Question: Which carrier-lane combinations have a source price at least double the lane's average source price, considering only non-cancelled loads?

SQL:
WITH lane_avg AS (
    SELECT
        lane_id,
        AVG(source_price) AS avg_lane_source_price
    FROM analytics.fact_loads
    WHERE load_was_cancelled = FALSE
    GROUP BY lane_id
)
SELECT
    f.carrier_id,
    c.carrier_name,
    f.lane_id,
    l.lane_name,
    f.loadsmart_id,
    f.source_price,
    la.avg_lane_source_price
FROM analytics.fact_loads f
JOIN lane_avg la
    ON f.lane_id = la.lane_id
JOIN analytics.dim_carrier c
    ON f.carrier_id = c.carrier_id
JOIN analytics.dim_lane l
    ON f.lane_id = l.lane_id
WHERE f.load_was_cancelled = FALSE
    AND f.source_price >= 2 * la.avg_lane_source_price
ORDER BY f.lane_id, f.carrier_id;

Answer: [{'carrier_id': '5f5ab1e815348370f5fcb1e5795f3e5b', 'carrier_name': 'Carrier 105736', 'lane_id': '121a6ad1aa39cca1d729cdab4d17ae2d', 'lane_name': 'Sioux City,IA -> Pittston,PA', 'loads